# 💾 Notebook 4 — The Saga Log: Surviving Crashes

Notebooks 1-3 showed the shape of a saga and how to write good compensations.
We kept one giant pretence the whole time: **the saga process never crashes**.

In production it will. The orchestrator pod restarts. The machine running choreography
loses power right after `charge()` but before emitting `PaymentCharged`. If the saga
state lived only in memory, you now have a customer charged for nothing and no way to
compensate — the saga forgot that step 2 even happened.

This notebook shows:

1. ❌ the **in-memory saga** — what's lost when the process dies
2. ✅ a **saga log**: every step's progress is written to durable storage *before* we act
3. 🔁 **crash recovery**: restart the saga, read the log, resume or compensate
4. 🏢 how real systems (Temporal, Step Functions, Camunda) implement the same idea

> **Key idea.** The saga's state machine must be **persisted**, not just executed.
> If it's not in a database (or event log), it didn't happen.

## 🛠️ Setup

```bash
cd 05-microservices/saga
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

We use only Python's standard library plus SQLite (built-in) as our "durable storage".

## 1. ❌ In-memory saga — what a crash destroys

Below, we start the checkout saga, charge the customer, then **simulate a crash** (`sys.exit`
replaced by a raised `SystemExit`). When the "replacement process" wakes up and tries to run
the saga again, it has no idea step 1 and 2 already ran — it re-reserves stock and re-charges
the customer. Double-charge, classic production incident.

In [ ]:
inventory = {"widget": 10}
payments  = {"charged": 0}

def reserve(order_id):
    inventory["widget"] -= 1
    print(f"  📦 reserved for order {order_id} (stock={inventory['widget']})")

def charge(order_id, amount):
    payments["charged"] += amount
    print(f"  💳 charged ${amount} for order {order_id} (total={payments['charged']})")

def checkout_in_memory(order_id, amount, crash_after_charge=False):
    # progress lives only in local variables — it dies with the process
    reserve(order_id)
    charge(order_id, amount)
    if crash_after_charge:
        raise SystemExit("💥 process crashed right after charge()")
    print("  🚚 ship... (not reached)")

# first attempt — crashes mid-way
try:
    checkout_in_memory(order_id=1, amount=20, crash_after_charge=True)
except SystemExit as e:
    print(e)

# the "replacement process" retries the same order — it has no memory of what happened
print("\n-- restart: retry same order 1 --")
checkout_in_memory(order_id=1, amount=20, crash_after_charge=False)

print("\n👉 final state:")
print("   inventory:", inventory, "(2 widgets gone for 1 order)")
print("   payments :", payments,  "(customer charged twice!)")

## 2. ✅ Add a saga log backed by SQLite

The fix is boringly mechanical: before each step, write `"about to do X"` to a log.
After the step succeeds, write `"X done"`. The log lives in a database that survives restarts.

On restart, the saga runner reads the log and:

- skips steps already marked `done` (don't double-charge),
- retries steps marked `started` (they may or may not have actually run — so each step
  must also be **idempotent**, as covered in notebook 3),
- continues from the next undone step.

We use SQLite because it ships with Python and needs zero setup. In production this is
Postgres, DynamoDB, or the event log of Kafka / Temporal.

In [ ]:
import sqlite3, os, tempfile, time

DB = os.path.join(tempfile.gettempdir(), "saga_log.db")
if os.path.exists(DB):
    os.remove(DB)  # start clean for a reproducible demo

def connect():
    conn = sqlite3.connect(DB)
    conn.execute("""
      CREATE TABLE IF NOT EXISTS saga_log (
        saga_id   TEXT    NOT NULL,
        step      TEXT    NOT NULL,
        status    TEXT    NOT NULL,           -- 'started' | 'done' | 'compensated'
        ts        REAL    NOT NULL,
        PRIMARY KEY (saga_id, step, status)
      )
    """)
    # The saga's own phase. Without this, a restart cannot tell
    # "I was moving forward" from "I was already unwinding" — and re-running the
    # forward steps of a compensating saga is how you double-charge someone.
    conn.execute("""
      CREATE TABLE IF NOT EXISTS saga_state (
        saga_id TEXT PRIMARY KEY,
        phase   TEXT NOT NULL                 -- 'running' | 'compensating' | 'done'
      )
    """)
    return conn

def log(saga_id, step, status):
    with connect() as conn:
        conn.execute(
            "INSERT OR IGNORE INTO saga_log(saga_id, step, status, ts) VALUES (?, ?, ?, ?)",
            (saga_id, step, status, time.time()),
        )

def steps_with_status(saga_id, status):
    with connect() as conn:
        rows = conn.execute(
            "SELECT step FROM saga_log WHERE saga_id=? AND status=?",
            (saga_id, status),
        ).fetchall()
    return {r[0] for r in rows}

def set_phase(saga_id, phase):
    with connect() as conn:
        conn.execute("INSERT INTO saga_state(saga_id, phase) VALUES (?, ?) "
                     "ON CONFLICT(saga_id) DO UPDATE SET phase=excluded.phase",
                     (saga_id, phase))

def get_phase(saga_id):
    with connect() as conn:
        row = conn.execute("SELECT phase FROM saga_state WHERE saga_id=?",
                           (saga_id,)).fetchone()
    return row[0] if row else 'running'

print("saga log ready at", DB)


### A tiny durable saga runner

Read it slowly — the 3 lines that matter are the `log(..., 'started')`, the call to `do()`,
and the `log(..., 'done')`. Everything else is plumbing.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class Step:
    name: str
    do: Callable[[], None]
    undo: Callable[[], None]

class Crash(BaseException):
    """Simulates the process dying.

    Deliberately derived from BaseException, NOT Exception, so the saga runner's
    `except Exception` does *not* catch it — exactly like a real SIGKILL, which
    gives your code no chance to run a compensation handler. Using a normal
    exception here would quietly turn 'the pod died' into 'the saga cleanly
    compensated', which is a completely different scenario."""

class DurableSaga:
    """Run a saga, persisting progress so we can resume after a crash."""
    def __init__(self, saga_id: str, steps: list[Step]):
        self.saga_id, self.steps = saga_id, steps

    def _compensate(self):
        """Undo every step that is 'done' and not yet 'compensated', in reverse."""
        done_set    = steps_with_status(self.saga_id, "done")
        already     = steps_with_status(self.saga_id, "compensated")
        for s in reversed(self.steps):
            if s.name not in done_set or s.name in already:
                continue
            print(f"  ↶ undo {s.name}")
            s.undo()
            log(self.saga_id, s.name, "compensated")
        set_phase(self.saga_id, "done")
        return "compensated"

    def run(self) -> str:
        # A saga that was already unwinding must NOT re-enter the forward path.
        if get_phase(self.saga_id) == "compensating":
            print("↷ resuming in COMPENSATING phase — skipping all forward steps")
            return self._compensate()

        set_phase(self.saga_id, "running")
        done_set    = steps_with_status(self.saga_id, "done")
        compensated = steps_with_status(self.saga_id, "compensated")
        for s in self.steps:
            if s.name in done_set and s.name not in compensated:
                print(f"↷ skip {s.name} — already done in a previous run")
                continue
            try:
                log(self.saga_id, s.name, "started")
                print(f"→ {s.name}")
                s.do()
                log(self.saga_id, s.name, "done")
            except Exception as e:
                print(f"✗ {s.name} failed: {e} — compensating")
                set_phase(self.saga_id, "compensating")   # durable BEFORE we unwind
                return self._compensate()
        set_phase(self.saga_id, "done")
        return "committed"


## 3. 🔁 Crash, restart, recover

We'll run the saga but have `charge` "crash" the process the **first time** it's called
(after it has updated the DB). Then we'll run the same saga id again — the runner reads
the log, sees `charge` is already `done`, and skips it. No double charge.

In [ ]:
inventory = {"widget": 10}
payments  = {"charged": 0, "refunds": 0}
shipping  = {"bookings": []}

SAGA_ID = "order-1"
crashed_once = {"charge": False}

def reserve_stock(): inventory["widget"] -= 1
def release_stock(): inventory["widget"] += 1

def charge():
    # The real business effect commits first...
    payments["charged"] += 20
    # ...and THEN the pod is killed, before the orchestrator can record 'done'.
    if not crashed_once["charge"]:
        crashed_once["charge"] = True
        raise Crash("💥 pod killed after charge committed")

def refund():       payments["refunds"] += 20
def book_courier(): shipping["bookings"].append("widget")
def cancel_courier():
    if "widget" in shipping["bookings"]: shipping["bookings"].remove("widget")

saga_steps = [
    Step("reserve_stock", reserve_stock, release_stock),
    Step("charge",        charge,        refund),
    Step("book_courier",  book_courier,  cancel_courier),
]

print("--- first run: the process DIES during charge ---")
try:
    DurableSaga(SAGA_ID, saga_steps).run()
except Crash as e:
    print(e)

print("\nstate after crash:")
print("  inventory:", inventory, "← still reserved: nothing was compensated,")
print("                             because a killed process compensates nothing")
print("  payments :", payments)
print("  log rows :", sorted(steps_with_status(SAGA_ID, 'started') |
                              steps_with_status(SAGA_ID, 'done')))
print("  phase    :", get_phase(SAGA_ID))


Notice the log now contains `charge → started` but **not** `charge → done`, and the
phase is still `running`. Nothing was compensated — a `SIGKILL` gives your process no
chance to run a handler, which is precisely why the log has to be durable.

A smart runner treats `started-but-not-done` as **uncertain**: the step may or may not
have actually run. It is the one state a saga log cannot resolve on its own.

Production systems resolve it in one of two ways:

- **Idempotency keys** — the participant service records the saga id; a repeat call is a
  no-op and returns the original result. Then "just retry it" is always safe.
- **Query-then-act** — on restart, ask the participant *"did you already process saga X?"*
  and act on its answer.

Below we do the second one against our (very simple) system of record: the payment
totals say the charge landed, so we upgrade that record to `done` and carry on.

In [ ]:
# Recovery step: reconcile 'started' entries with the real system of record.
# In production: call the payment service with the saga id to check status.
# Here we know the charge went through (payments['charged'] == 20), so mark it done.
started = steps_with_status(SAGA_ID, "started")
done    = steps_with_status(SAGA_ID, "done")
uncertain = started - done
print("uncertain steps from last run:", uncertain)

if "charge" in uncertain and payments["charged"] >= 20:
    log(SAGA_ID, "charge", "done")
    print("  reconciled: charge actually succeeded → marked done")

print("\n--- second run: resume the same saga id ---")
result = DurableSaga(SAGA_ID, saga_steps).run()

print("\nresult   :", result)
print("inventory:", inventory)
print("payments :", payments, "← still charged once, NOT twice ✅")
print("shipping :", shipping)

## 4. What if the crash happens *during* compensation?

This is the case that breaks naive implementations. Below, `book_courier` fails, the
saga starts unwinding, refunds the charge — and then the pod dies before it can release
the stock.

On restart, a runner that only looks at `done`/`compensated` rows faces an impossible
question: `reserve_stock` is `done` and `charge` is `done`, so should it go *forward*?
If it does, it re-runs `charge` and the customer pays twice.

The fix is the `phase` column we added: the saga durably records that it switched to
`compensating` **before** it ran a single `undo`. On restart it reads that and resumes
unwinding instead of marching forward.

In [ ]:
inventory = {"widget": 10}
payments  = {"charged": 0, "refunds": 0}
shipping  = {"bookings": []}
SAGA_ID2 = "order-2"

comp_crashed = {"release_stock": False}

def reserve2(): inventory["widget"] -= 1
def release2():
    # 💥 the pod dies the FIRST time we try to compensate stock
    if not comp_crashed["release_stock"]:
        comp_crashed["release_stock"] = True
        raise Crash("💥 pod killed during compensation")
    inventory["widget"] += 1
def charge2(): payments["charged"] += 20
def refund2(): payments["refunds"] += 20
def ship_fail(): raise RuntimeError("courier API timeout")
def noop(): pass

steps2 = [
    Step("reserve_stock", reserve2,  release2),
    Step("charge",        charge2,   refund2),
    Step("book_courier",  ship_fail, noop),
]

print("--- run 1: ship fails → compensate → pod dies inside release_stock ---")
try:
    DurableSaga(SAGA_ID2, steps2).run()
except Crash as e:
    print(e)
print("state after crash:")
print("  inventory:", inventory, "| payments:", payments)
print("  compensated so far:", steps_with_status(SAGA_ID2, 'compensated'))
print("  phase             :", get_phase(SAGA_ID2), "← this is what saves us")

print("\n--- run 2: resume ---")
print("Because the phase is 'compensating', the runner goes straight back to")
print("unwinding. It does NOT re-run book_courier, and — critically — it does not")
print("re-run charge, which would have charged the customer a second time.")
DurableSaga(SAGA_ID2, steps2).run()

print("\ninventory:", inventory, "← stock back to 10 ✅")
print("payments :", payments,  "← refunded exactly once ✅")


## 5. 🏢 How real systems do this

You just built — in ~50 lines of SQLite — the core of every production saga engine.
The big tools add durability guarantees, retries, timers, and observability on top:

| Tool                       | Where the saga log lives                        | What it adds                                  |
|---------------------------|--------------------------------------------------|-----------------------------------------------|
| **Temporal / Cadence**    | Its own event-sourced history DB                 | Durable timers, auto-retry, versioning, UI    |
| **AWS Step Functions**    | AWS-managed state store                          | Visual state machine, catch/retry DSL         |
| **Camunda / Zeebe**       | Relational DB or Zeebe log                       | BPMN modelling, human tasks, incidents UI     |
| **Netflix Conductor**     | Cassandra / Dynomite / Postgres                  | Workflow versioning, workers in many languages|
| **Kafka + outbox**        | Kafka topic + per-service DB outbox table        | Choreography with exactly-once-ish delivery   |

The rules you met in notebooks 1–3 still apply on top of any of these:

- Every step must be **idempotent** (section 1 of notebook 3).
- Distinguish **retryable vs non-retryable** errors (section 2 of notebook 3).
- Use **semantic compensations** when you can't literally undo (section 3 of notebook 3).
- Identify the **pivot transaction**; after it, go forward-only (section 4 of notebook 3).

The saga log simply makes sure your orchestrator / choreography can answer one question
after any crash: **"what have I already done for this saga id?"**. Everything else follows.

## ✅ Recap

- An in-memory saga is **one crash away** from a broken business state.
- A **saga log** persists `(saga_id, step, status)` before and after each action.
- On restart, the runner skips `done` steps, reconciles `started-but-not-done` steps with
  the real participants, and resumes.
- The log must also record the saga **phase** (`running` / `compensating`). Without it a
  saga that crashed mid-unwind will happily march forward again and re-execute steps it
  had already decided to undo.
- Participants still need **idempotency** — the saga log protects the orchestrator, not them.
- Real tools (Temporal, Step Functions, Camunda) give you this out of the box, plus timers,
  retries, and dashboards.

### Try next
- Replace SQLite with Postgres and run two orchestrator processes — add a `lease` column so
  only one can own a saga id at a time.
- Add timeouts: if a step's `started` row is older than N seconds, auto-compensate.
- Combine with the **outbox pattern** (`04-patterns/outbox-and-cdc/`) to publish events
  atomically with the saga log update.